In [1]:
import numpy as np
import pandas as pd

import os

In [2]:
import json

with open("../data/ru_ecomm_tree_category.json", encoding="utf-8") as file:
    data_categories = json.load(file)

In [3]:
labeled_data = pd.read_csv("../data/raw/mlops_dataset.tsv", sep='\t')

In [4]:
labeled_data = labeled_data.dropna()
labeled_data = labeled_data.drop_duplicates(keep="first")
labeled_data = labeled_data.reset_index(drop=True)

In [5]:
labeled_data

,url,moderation_category_name
0,https://www.orgmebel.ru/product/rabochaya-stan...,Электроника -> Офисная техника
1,https://www.pult.ru/product/interfeys-izbytoch...,Электроника -> Аксессуары для электроники
2,https://bellavka.ru/model/moda-jurs-2879-haki-...,"Одежда, обувь и аксессуары -> Женская одежда -..."
3,https://apteka.ru/product/65a921ee0c42f5a5dec7...,Здоровье -> Витамины и БАДы
4,https://luxkatalog.ru/p-zara-polyamide-bodysui...,"Одежда, обувь и аксессуары -> Женская одежда -..."
...,...,...
49219,https://pushe.ru/product/83073662/,Мебель -> Столы и стулья
49220,https://novokuznetsk.e2e4online.ru/catalog/ite...,Электроника -> Офисная техника
49221,https://www.mvideo.ru/products/pogruzhnoi-blen...,Товары для дома -> Посуда и кухонные принадлеж...
49222,https://www.dolina-podarkov.ru/product/3879,Продукты питания -> Сладости


In [6]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}


def extract_product_info(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
    except Exception as e:
        print(f"Failed to load URL: {e}")
        return None

    # Title extraction
    title = (
        soup.find("meta", property="og:title") or
        soup.find("meta", attrs={"name": "title"}) or
        soup.title
    )
    title = title.get("content") if title and title.has_attr("content") else getattr(title, "text", "").strip()

    # Description extraction
    description = (
        soup.find("meta", property="og:description") or
        soup.find("meta", attrs={"name": "description"})
    )
    description = description.get("content") if description else ""

    # Image extraction (try og:image first)
    image = soup.find("meta", property="og:image")
    image_url = image.get("content") if image else ""

    if not image_url:
        # Fallback to largest visible image
        images = soup.find_all("img")
        image_candidates = []
        for img in images:
            src = img.get("src")
            if not src or "logo" in src or "icon" in src:
                continue
            full_url = urljoin(url, src)
            width = int(img.get("width", 0)) if img.get("width", "").isdigit() else 0
            height = int(img.get("height", 0)) if img.get("height", "").isdigit() else 0
            image_candidates.append((width * height, full_url))

        if image_candidates:
            image_candidates.sort(reverse=True)  # biggest first
            image_url = image_candidates[0][1]

    return {
        "url": url,
        "title": title.strip(),
        "description": description.strip(),
        "image_url": image_url.strip()
    }

In [7]:
parsed_data = []

In [8]:
records = labeled_data.to_dict(orient="records")


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_record(idx, record):
    url = record["url"]
    category = record["moderation_category_name"]
    data = {
        "url": url,
        "title": None,
        "description": None,
        "image_url": None,
        "category": category
    }
    info_by_url = extract_product_info(url)
    if info_by_url:
        data["title"] = info_by_url.get("title")
        data["description"] = info_by_url.get("description")
        data["image_url"] = info_by_url.get("image_url")
    print(f"{idx} {info_by_url is not None} Get info for {url}")
    return data


# Максимальное количество потоков — подберите по вашей нагрузке (например, 10 или 20)
with ThreadPoolExecutor(max_workers=50) as executor:
    # Запускаем задачи
    futures = [executor.submit(process_record, idx, record) for idx, record in enumerate(records)]

    for future in as_completed(futures):
        result = future.result()
        parsed_data.append(result)


In [3]:
cleaned_df = pd.read_csv("cleaned_result_parsing.csv", encoding="utf-8")

In [5]:
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27878 entries, 0 to 27877
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   url          27878 non-null  object
 1   title        27877 non-null  object
 2   description  27011 non-null  object
 3   image_url    27873 non-null  object
 4   category     27878 non-null  object
dtypes: object(5)
memory usage: 1.1+ MB


In [7]:
import pandas as pd
import numpy as np
from PIL import Image
from io import BytesIO
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Функция для загрузки и преобразования изображения
def fetch_image_np(url):
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
        return np.array(image)
    except Exception:
        return None

# Обёртка для tqdm + многопоточности
def fetch_images_parallel(urls, max_workers=32):
    results = [None] * len(urls)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_image_np, url): i for i, url in enumerate(urls)}
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Загрузка изображений"):
            idx = futures[future]
            results[idx] = future.result()

    return results

In [8]:
# Распараллеливаем
image_arrays = fetch_images_parallel(cleaned_df["image_url"].tolist(), max_workers=32)

# Добавляем в DataFrame и сохраняем
cleaned_df["image_np"] = image_arrays
cleaned_df.to_pickle("dataset_with_images.pkl")

Загрузка изображений:   5%|▌         | 1497/27878 [00:54<15:53, 27.67it/s]  


KeyboardInterrupt: 

/home/tellowit/.config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:900: UserWarning: Corrupt EXIF data.  Expecting to read 12 bytes but only got 0. 
  warnings.warn(str(msg))
/home/tellowit/.config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
